<a href="https://colab.research.google.com/github/Durga22-amie/-vqe-cancer-segmentation-/blob/main/Brain_MRI_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
masoudnickparvar_brain_tumor_mri_dataset_path = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

print(os.listdir("/kaggle/input"))

['datasets']


In [ ]:
dataset_path = "/kaggle/input/datasets"

print(os.listdir(dataset_path))

['masoudnickparvar']


In [ ]:
import os

base = "/kaggle/input/datasets/masoudnickparvar"

print(os.listdir(base))

['brain-tumor-mri-dataset']


In [ ]:
import os

base = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"

print(os.listdir(base))

['Training', 'Testing']


In [ ]:
print("Training:")
print(os.listdir(base + "/Training"))

print("\nTesting:")
print(os.listdir(base + "/Testing"))

Training:
['pituitary', 'notumor', 'meningioma', 'glioma']

Testing:
['pituitary', 'notumor', 'meningioma', 'glioma']


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

base = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"

train_dir = os.path.join(base, "Training")
test_dir = os.path.join(base, "Testing")

classes = {
    "glioma":0,
    "meningioma":1,
    "notumor":2,
    "pituitary":3
}

images = []
labels = []

for folder, label in classes.items():

    for split in [train_dir, test_dir]:

        folder_path = os.path.join(split, folder)

        for file in tqdm(os.listdir(folder_path)):

            path = os.path.join(folder_path, file)

            try:
                img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

                img = cv2.resize(img, (128,128))

                images.append(img)
                labels.append(label)

            except:
                pass

images = np.array(images)
labels = np.array(labels)

print(images.shape)
print(labels.shape)

100%|██████████| 400/400 [00:03<00:00, 101.02it/s]


(7200, 128, 128)
(7200,)


In [ ]:
from skimage.feature import graycomatrix, graycoprops

results = []

for img,label in tqdm(zip(images,labels),
                      total=len(images)):

    mean = np.mean(img)
    std = np.std(img)
    variance = np.var(img)

    energy = np.sum(img.astype(np.float64)**2)

    hist,_ = np.histogram(
        img,
        bins=64,
        density=True
    )

    hist = hist[hist>0]

    entropy = -np.sum(hist*np.log2(hist))

    glcm = graycomatrix(
        img,
        distances=[1],
        angles=[0],
        levels=256,
        symmetric=True,
        normed=True
    )

    contrast = graycoprops(
        glcm,
        "contrast"
    )[0,0]

    correlation = graycoprops(
        glcm,
        "correlation"
    )[0,0]

    homogeneity = graycoprops(
        glcm,
        "homogeneity"
    )[0,0]

    dissimilarity = graycoprops(
        glcm,
        "dissimilarity"
    )[0,0]

    asm = graycoprops(
        glcm,
        "ASM"
    )[0,0]

    results.append([
        mean,
        std,
        variance,
        energy,
        entropy,
        contrast,
        correlation,
        homogeneity,
        dissimilarity,
        asm,
        label
    ])

100%|██████████| 7200/7200 [00:24<00:00, 289.52it/s]


In [ ]:
columns = [
    "Mean",
    "Std",
    "Variance",
    "Energy",
    "Entropy",
    "Contrast",
    "Correlation",
    "Homogeneity",
    "Dissimilarity",
    "ASM",
    "Label"
]

df = pd.DataFrame(results,
                  columns=columns)

print(df.shape)

df.head()

(7200, 11)


,Mean,Std,Variance,Energy,Entropy,Contrast,Correlation,Homogeneity,Dissimilarity,ASM,Label
0,25.472839,25.012731,625.636713,20881445.0,1.669309,144.518086,0.884541,0.432868,6.016117,0.019698,0
1,16.507996,23.052318,531.409360,13171479.0,1.340970,87.535679,0.917950,0.657992,3.179380,0.101926,0
2,38.106689,45.992236,2115.285737,58448372.0,1.590835,370.616326,0.912647,0.430276,8.644624,0.020097,0
3,23.067505,31.076162,965.727865,24540572.0,1.284502,190.338583,0.901797,0.662070,4.911171,0.133729,0
4,33.205078,31.257289,977.018124,34072090.0,1.809177,224.710568,0.884926,0.449357,7.091228,0.028430,0


In [ ]:
X = df.drop(columns=["Label"])

y = df["Label"]

top8 = X.var().sort_values(
    ascending=False
).head(8)

print(top8)

selected = top8.index.tolist()

X = X[selected]

Energy           2.944111e+15
Variance         1.939586e+06
Contrast         2.866819e+05
Mean             3.163604e+02
Std              1.545326e+02
Dissimilarity    2.245634e+01
Entropy          3.564367e-02
Homogeneity      1.785756e-02
dtype: float64


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(7200, 8)


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

indices = np.arange(len(X_scaled))

sample_idx, _ = train_test_split(
    indices,
    train_size=1000,
    stratify=y,
    random_state=42
)

X_scaled_small = X_scaled[sample_idx]

y_small = y.iloc[sample_idx].reset_index(drop=True)

print("X shape:", X_scaled_small.shape)
print("y shape:", y_small.shape)
print("Class distribution:")
print(np.bincount(y_small))

X shape: (1000, 8)
y shape: (1000,)
Class distribution:
[250 250 250 250]


In [ ]:
from qiskit.quantum_info import SparsePauliOp

def create_hamiltonian(features):

    paulis = []
    coeffs = []

    n_qubits = len(features)

    for i, value in enumerate(features):

        label = ["I"] * n_qubits
        label[i] = "Z"

        paulis.append("".join(label))
        coeffs.append(float(value))

    return SparsePauliOp(paulis, coeffs)

In [ ]:
!pip install qiskit qiskit-algorithms qiskit-aer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 64.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 85.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.8 MB/s eta 0:00:00


In [ ]:
from qiskit.quantum_info import SparsePauliOp

def create_hamiltonian(features):
    """
    Convert radiomic features into a Hamiltonian.

    Parameters
    ----------
    features : array-like
        Normalized feature vector.

    Returns
    -------
    SparsePauliOp
        Hamiltonian H = Σ xi Zi
    """

    paulis = []
    coeffs = []

    n_qubits = len(features)

    for i, value in enumerate(features):

        label = ["I"] * n_qubits
        label[i] = "Z"

        paulis.append("".join(label))
        coeffs.append(float(value))

    return SparsePauliOp(paulis, coeffs)

In [ ]:
H = create_hamiltonian(X_scaled[0])

print(H)

SparsePauliOp(['ZIIIIIII', 'IZIIIIII', 'IIZIIIII', 'IIIZIIII', 'IIIIZIII', 'IIIIIZII', 'IIIIIIZI', 'IIIIIIIZ'],
              coeffs=[0.02604947+0.j, 0.02107508+0.j, 0.01801804+0.j, 0.08535193+0.j,
 0.05508175+0.j, 0.07110786+0.j, 0.30281512+0.j, 0.55678456+0.j])


In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.circuit.library import RealAmplitudes

estimator = StatevectorEstimator()

optimizer = COBYLA(maxiter=50)

ansatz = RealAmplitudes(
    num_qubits=8,
    reps=2
)

vqe = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=optimizer
)

/tmp/ipykernel_58/1515839546.py:10: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(


In [ ]:
H = create_hamiltonian(X_scaled_small[0])

result = vqe.compute_minimum_eigenvalue(H)

print(result.eigenvalue.real)

-0.3651104420553421


In [ ]:
print(result.eigenvalue.real)

-0.3651104420553421


In [ ]:
from tqdm import tqdm

vqe_energy = []

for i in tqdm(range(len(X_scaled_small))):

    H = create_hamiltonian(X_scaled_small[i])

    result = vqe.compute_minimum_eigenvalue(H)

    vqe_energy.append(result.eigenvalue.real)

100%|██████████| 1000/1000 [05:28<00:00,  3.04it/s]


In [ ]:
import pandas as pd

X_quantum = pd.DataFrame(
    X_scaled_small,
    columns=selected
)

X_quantum["VQE"] = vqe_energy

print(X_quantum.head())

     Energy  Variance  Contrast      Mean       Std  Dissimilarity   Entropy  \
0  0.057220  0.101059  0.039403  0.084708  0.209120       0.054522  0.134388   
1  0.101849  0.129444  0.050105  0.187369  0.252891       0.116318  0.262278   
2  0.492616  0.657995  0.173362  0.499527  0.772737       0.267503  0.261251   
3  0.166558  0.273426  0.086556  0.207892  0.433917       0.130370  0.141356   
4  0.601540  0.539808  0.220913  0.672950  0.681603       0.350382  0.402374   

   Homogeneity       VQE  
0     0.855229 -0.964780  
1     0.594475 -1.270309  
2     0.543811 -2.431642  
3     0.757879 -1.492678  
4     0.190948 -2.801894  


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

classical_scores = []

for train, test in cv.split(X_scaled_small, y_small):

    clf = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    clf.fit(
        X_scaled_small[train],
        y_small.iloc[train]
    )

    pred = clf.predict(
        X_scaled_small[test]
    )

    classical_scores.append(
        accuracy_score(
            y_small.iloc[test],
            pred
        )
    )

print("Classical Scores:")
print(classical_scores)

print("Mean:",
      np.mean(classical_scores))

Classical Scores:
[0.74, 0.71, 0.78, 0.76, 0.76, 0.72, 0.86, 0.75, 0.79, 0.69]
Mean: 0.756


In [ ]:
quantum_scores = []

for train, test in cv.split(X_quantum, y_small):

    clf = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    clf.fit(
        X_quantum.iloc[train],
        y_small.iloc[train]
    )

    pred = clf.predict(
        X_quantum.iloc[test]
    )

    quantum_scores.append(
        accuracy_score(
            y_small.iloc[test],
            pred
        )
    )

print("Quantum Scores:")
print(quantum_scores)

print("Mean:",
      np.mean(quantum_scores))

Quantum Scores:
[0.75, 0.7, 0.76, 0.76, 0.73, 0.72, 0.79, 0.74, 0.81, 0.65]
Mean: 0.741


In [ ]:
from scipy.stats import ttest_rel, wilcoxon

t_stat, p_t = ttest_rel(
    classical_scores,
    quantum_scores
)

w_stat, p_w = wilcoxon(
    classical_scores,
    quantum_scores
)

print("Classical Mean:",
      np.mean(classical_scores))

print("Quantum Mean:",
      np.mean(quantum_scores))

print("Paired t-test p-value:",
      p_t)

print("Wilcoxon p-value:",
      p_w)

Classical Mean: 0.756
Quantum Mean: 0.741
Paired t-test p-value: 0.10539067158640898
Wilcoxon p-value: 0.1328125
